# Import packages to be used

In [25]:
# Import packages
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import matplotlib_inline.backend_inline
import nbformat

## Create dataframes from raw csv files


In [26]:
# Create DataFrames from the CSVs
customers_df = pd.read_csv('customers.csv')
orders_df = pd.read_csv('orders.csv')
products_df = pd.read_csv('products.csv')

#Display the first few rows of each DataFrame
print("Customers:")
display(customers_df.head())
print("Products:")
display(products_df.head())
print("Orders:")
display(orders_df.head())


Customers:


,customer_id,first_name,last_name,email,city,region,country,signup_date,age,is_subscribed,loyalty_tier
0,CUST-0001,James,Brown,james.brown@gmail.com,London,England,United Kingdom,2022-01-31,65,True,Bronze
1,CUST-0002,Emily,Campbell,emily.campbell@outlook.com,Birmingham,England,United Kingdom,2023-09-25,55,True,Bronze
2,CUST-0003,Arthur,Roberts,arthur.roberts@icloud.com,London,England,United Kingdom,2021-03-25,53,False,Bronze
3,CUST-0004,Freddie,Murphy,freddie.murphy@outlook.com,London,England,United Kingdom,2020-10-15,66,False,Bronze
4,CUST-0005,Leo,Stewart,leo.stewart@hotmail.com,Glasgow,Scotland,United Kingdom,2020-01-15,24,False,Bronze


Products:


,product_id,product_name,category,price,cost,stock_quantity,supplier
0,PRD_001,Smart Webcam,Electronics,125.21,91.76,322,TD SYNNEX
1,PRD_002,Smart Speaker,Electronics,336.32,191.26,469,TD SYNNEX
2,PRD_003,Rechargeable Speaker,Electronics,150.81,79.02,574,Tech Data
3,PRD_004,Wireless Router,Electronics,149.79,93.68,597,Exertis
4,PRD_005,4K Monitor,Electronics,196.94,147.55,141,Tech Data


Orders:


,order_id,customer_id,product_id,order_date,quantity,unit_price,total_amount,order_status,payment_method
0,ORD-000001,CUST-0251,PRD_328,2023-02-21,1,27.96,27.96,Completed,Credit Card
1,ORD-000002,CUST-0033,PRD_347,2023-06-28,5,51.74,258.70,Completed,Debit Card
2,ORD-000003,CUST-0734,PRD_120,2023-02-24,5,96.94,484.70,Completed,Google Pay
3,ORD-000004,CUST-0829,PRD_215,2025-07-08,2,35.22,70.44,Pending,Debit Card
4,ORD-000005,CUST-0221,PRD_358,2024-11-27,4,10.92,43.68,Returned,PayPal


## Analysing some customer data

Using pandas for data manipulation and plotly to display results.

In [ ]:
# Create a Plotly bar chart for total customers by region
region_counts = customers_df['region'].value_counts().sort_index()
fig1 = px.bar(
    x=region_counts.index,
    y=region_counts.values,
    labels={'x': 'Region', 'y': 'Total Customers'},
    title='Total Customers by Region',
    text=region_counts.values,
    color=region_counts.index,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig1.update_traces(texttemplate='%{text}', textposition='inside', textfont_color='white')
fig1.update_layout(
    xaxis_title='',
    yaxis_title='',
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig1.show()

# create a grouped bar chart of age distribution by region
bins = [18, 30, 50, 100]
labels = ["18-29", "30-49", "50+"]
customers_df['age_group'] = pd.cut(
    customers_df['age'], bins=bins, labels=labels, right=False
    )
age_region_counts = customers_df.groupby(['region', 'age_group']).size().reset_index(name='count')
fig2 = px.bar(
    age_region_counts,
    x='region',
    y='count',
    color='age_group',
    barmode='group',
    labels={'region': 'Region', 'count': 'Customer Count', 'age_group': 'Age Range'},
    title='Age Distribution of Customers by Region',
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig2.update_layout(
    xaxis_title='Region',
    yaxis_title='Customer Count',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig2.show()

# create a pie chart for overall age distribution of customers
age_counts = customers_df['age_group'].value_counts().sort_index()
fig3 = px.pie(
    values=age_counts.values,
    names=age_counts.index,
    title='Age Distribution of All Customers',
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig3.update_traces(textinfo='percent+label', textfont_color='white')
fig3.update_layout(
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig3.show()

# create a grouped bar chart for age range vs loyalty tier and region (percentage)

tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
tier_colors = {'Bronze': '#cd7f32', 'Silver': '#c0c0c0', 'Gold': '#ffd700', 'Platinum': '#e5e4e2'}
tier_counts = customers_df.groupby(['region', 'age_group', 'loyalty_tier']).size().reset_index(name='count')
total_by_age_region = customers_df.groupby(['region', 'age_group']).size().reset_index(name='total')
merged = pd.merge(tier_counts, total_by_age_region, on=['region', 'age_group'])
merged['percent'] = merged['count'] / merged['total'] * 100
merged['loyalty_tier'] = pd.Categorical(merged['loyalty_tier'], categories=tier_order, ordered=True)
fig4 = px.bar(
    merged.sort_values('loyalty_tier'),
    x='region',
    y='percent',
    color='loyalty_tier',
    facet_col='age_group',
    barmode='stack',
    labels={'region': 'Region', 'percent': 'Percent of Loyalty Tier', 'loyalty_tier': 'Loyalty Tier', 'age_group': 'Age Range'},
    title='Percentage of Loyalty Tier in Each Age Group by Region',
    color_discrete_map=tier_colors
    )
fig4.update_layout(
    xaxis_title='',
    yaxis_title='Percent',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig4.show()


## Analysing Products
